# Project 2: AI-Driven Phishing Email Detection Using NLP

This notebook implements a machine learning system that automatically detects phishing emails using text analysis and natural language processing (NLP).

## Phase 1: Data Collection
Gather phishing and legitimate email samples. If a dataset is not found in the `Dataset` folder, a small synthetic dataset will be generated to demonstrate the pipeline.

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATASET_PATH = "../Dataset/phishing_email_dataset.csv"

if os.path.exists(DATASET_PATH):
    print(f"Loading dataset from {DATASET_PATH}")
    data = pd.read_csv(DATASET_PATH)
else:
    print("Dataset not found. Generating a synthetic mock dataset for demonstration purposes...")
    mock_data = {
        'email_body': [
            "Dear customer, your account has been compromised. Click here to verify: http://fake-login.com",
            "Hi team, attached is the report for this week's sprint. Best, Alice",
            "URGENT: You have won a $1000 gift card! Claim now by providing your credit card details.",
            "Let's schedule a meeting for tomorrow at 10 AM to discuss the project timeline.",
            "Your PayPal account is restricted. <a href='http://paypal-update-info.com'>Update here</a> to restore access.",
            "Can we postpone our lunch to Friday? I have a conflict today.",
            "You have 1 new secure message from Bank of America. Log in to view.",
            "Here are the notes from yesterday's presentation. Let me know if you need any edits.",
            "Final warning: your Office 365 password expires today. Validate your identity immediately.",
            "Happy birthday! Hope you have a wonderful day surrounded by friends and family."
        ] * 50, # Multiply to create enough data for splitting and training
        'label': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0] * 50 # 1 = Phishing, 0 = Legitimate
    }
    data = pd.DataFrame(mock_data)

print(f"Dataset shape: {data.shape}")
data.head()

## Phase 2: Data Cleaning
Remove HTML tags, punctuation, stopwords, and normalize text.

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def clean_email(text):
    text = str(text)
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove URLs (replace with a placeholder 'URL')
    text = re.sub(r'http[s]?://\S+', 'URL', text)
    # Remove punctuation & non-alphanumeric
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Lowercase
    text = text.lower()
    # Remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

data['cleaned_text'] = data['email_body'].apply(clean_email)
print("Sample original:\n", data['email_body'].iloc[0])
print("\nSample cleaned:\n", data['cleaned_text'].iloc[0])

## Phase 3: Feature Engineering
Apply TF-IDF and extract metadata features (e.g., presence of URLs or urgency words).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = data['cleaned_text']
y = data['label']

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=3000)
X_vec = vectorizer.fit_transform(X)

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

## Phase 4: Model Development
Train and compare models: Logistic Regression, Random Forest, Naive Bayes, and a Simple Neural Network.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(50,), max_iter=300, random_state=42)
}

## Phase 5: Evaluation
Compare models using accuracy, precision, recall, and F1-score. Visualize confusion matrices.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

results = []

for name, model in models.items():
    print(f"\n{'='*40}\nEvaluating {name}\n{'='*40}")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
    plt.title(f"Confusion Matrix - {name}")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

In [ ]:
# Summary DataFrame and Comparison Plot
results_df = pd.DataFrame(results)
display(results_df)

results_df.set_index('Model').plot(kind='bar', figsize=(10, 5), colormap='Set2')
plt.title('Algorithm Comparison across Metrics')
plt.ylim(0, 1.1)
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (for Random Forest)
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_
feature_names = vectorizer.get_feature_names_out()

indices = np.argsort(importances)[::-1][:20] # Top 20 features

plt.figure(figsize=(10, 6))
plt.title("Top 20 Feature Importances (Random Forest)")
plt.bar(range(20), importances[indices], align="center", color='teal')
plt.xticks(range(20), feature_names[indices], rotation=45, ha='right')
plt.xlim([-1, 20])
plt.tight_layout()
plt.show()